# Phase 6 — CI/CD, Deployment Validation, Rollback, and Final Report

**Project:** Enterprise Customer Support Automation with Agentic RAG  
**Approach:** GitHub Actions + pytest + Ruff + deployment health checks  
**Containerization is intentionally excluded**, as requested.

## What this notebook creates
- `.github/workflows/ci.yml` — automated lint and tests on pushes and pull requests.
- `.github/workflows/deployment-validation.yml` — manually triggered validation of an already-deployed API.
- `tests/test_api.py` — fast API contract tests that do not download ML models.
- `scripts/validate_deployment.py` — checks a deployed `/health` endpoint.
- `reports/phase6_rollback_and_release_plan.md` — beginner-friendly release/rollback plan.
- `reports/phase6_final_evidence_checklist.md` — checklist for required screenshots and final submission.
- `reports/phase6_project_report_template.md` — project report structure aligned to the capstone brief.

**Important:** GitHub Actions CI will test code; it will not magically publish your local app to the internet. The deployment validation workflow checks a URL you deploy using your chosen hosting provider. No hosting provider has been selected in this project, so the notebook does not pretend to deploy to one.

Run this notebook from the `notebooks` folder, using the same `.venv` kernel. Execute all cells from top to bottom.

## 1. Locate the project root

The notebook searches upward for the Phase 5 files. If it cannot find them, set `PROJECT_ROOT` manually to your `capstone-rag-agents` folder.

In [ ]:
from pathlib import Path
import os

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (candidate / "src" / "api.py").exists() and (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    # If auto-detection fails, uncomment and edit this line:
    # PROJECT_ROOT = Path(r"C:\Users\YOUR_USER\OneDrive - ENCORA\Desktop\capstone-rag-agents")
    raise FileNotFoundError(
        "Could not find src/api.py and requirements.txt. Open this notebook from your project's notebooks folder."
    )

print("Project root:", PROJECT_ROOT)
print("API file exists:", (PROJECT_ROOT / "src" / "api.py").exists())
print("Requirements file exists:", (PROJECT_ROOT / "requirements.txt").exists())

## 2. Create the folders required for CI/CD

In [ ]:
from pathlib import Path

WORKFLOW_DIR = PROJECT_ROOT / ".github" / "workflows"
TEST_DIR = PROJECT_ROOT / "tests"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
REPORT_DIR = PROJECT_ROOT / "reports"
for folder in [WORKFLOW_DIR, TEST_DIR, SCRIPTS_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Created/verified .github/workflows, tests, scripts, and reports folders.")

## 3. Create a fast, model-free API test suite

The tests check API contracts and input validation. They deliberately do **not** call `/query` with a valid question because that would load/download the embedding and generation models and make CI slow and network-dependent. We will still check `/query` input validation.

In [ ]:
test_code = r"""
from pathlib import Path
import sys

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fastapi.testclient import TestClient
import api

client = TestClient(api.app)


def test_health_endpoint_returns_ok():
    response = client.get("/health")
    assert response.status_code == 200
    body = response.json()
    assert body["status"] == "ok"
    assert isinstance(body["chunks_file_exists"], bool)
    assert isinstance(body["faiss_index_exists"], bool)


def test_query_rejects_empty_question():
    response = client.post("/query", json={"question": "", "top_k": 4})
    assert response.status_code == 422


def test_query_rejects_top_k_out_of_range():
    response = client.post("/query", json={"question": "Where is my order?", "top_k": 100})
    assert response.status_code == 422


def test_feedback_rejects_invalid_rating():
    response = client.post(
        "/feedback",
        json={
            "question": "Test question",
            "answer": "Test answer",
            "rating": "neutral",
            "comment": "Should fail schema validation",
        },
    )
    assert response.status_code == 422


def test_feedback_endpoint_accepts_valid_feedback(monkeypatch):
    # Avoid writing a real feedback file during automated tests.
    monkeypatch.setattr(
        api,
        "save_feedback",
        lambda **kwargs: {"status": "saved", "rating": kwargs["rating"]},
    )
    response = client.post(
        "/feedback",
        json={
            "question": "Where is my order?",
            "answer": "Use the order tracking page.",
            "rating": "positive",
            "comment": "Test feedback",
        },
    )
    assert response.status_code == 200
    assert response.json()["status"] == "saved"
    assert response.json()["rating"] == "positive"
"""
(TEST_DIR / "test_api.py").write_text(test_code.strip() + "\n", encoding="utf-8")
print("Created model-free API tests:", TEST_DIR / "test_api.py")

## 4. Configure Ruff for a practical first CI check

Ruff is a Python linter. To avoid blocking this beginner project on cosmetic style issues, CI initially checks important syntax/name errors. You can tighten lint rules after the project is stable.

In [ ]:
ruff_config = r"""
line-length = 100

[lint]
select = ["E9", "F63", "F7", "F82"]
"""
(PROJECT_ROOT / "pyproject.toml").write_text(ruff_config.strip() + "\n", encoding="utf-8")
print("Created:", PROJECT_ROOT / "pyproject.toml")

## 5. Create the GitHub Actions CI workflow

This workflow runs on every push and pull request. It uses Python 3.11 because the ML dependencies in this project may not all be ready for Python 3.14. It installs dependencies, runs Ruff, and runs pytest. It does **not** build or publish a container image.

Before pushing, make sure the project files (`src/`, `tests/`, `requirements.txt`, `pyproject.toml`, and the small synthetic Phase 1/2 artifacts needed for local demos) are committed. Never commit API keys, `.venv`, or private data.

In [ ]:
ci_workflow = r"""
name: Customer Support RAG CI

on:
  push:
    branches: [main, master, develop]
  pull_request:
    branches: [main, master, develop]
  workflow_dispatch:

permissions:
  contents: read

jobs:
  test:
    runs-on: ubuntu-latest
    timeout-minutes: 30
    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
          cache-dependency-path: requirements.txt

      - name: Upgrade pip
        run: python -m pip install --upgrade pip

      - name: Install project dependencies
        run: python -m pip install -r requirements.txt ruff

      - name: Lint critical Python errors
        run: ruff check src tests

      - name: Run automated API tests
        env:
          PYTHONPATH: src
        run: python -m pytest -q
"""
(WORKFLOW_DIR / "ci.yml").write_text(ci_workflow.strip() + "\n", encoding="utf-8")
print("Created:", WORKFLOW_DIR / "ci.yml")

## 6. Create a deployment health-check script

This script verifies a deployment that already exists. It requests the `/health` endpoint, checks the HTTP status and JSON response, and returns a non-zero exit code if validation fails.

In [ ]:
validation_script = r"""
import json
import os
import sys
from urllib.parse import urlparse

import requests


def main() -> int:
    base_url = os.getenv("DEPLOY_HEALTH_URL", "").strip().rstrip("/")
    if not base_url:
        print("ERROR: Set DEPLOY_HEALTH_URL to the deployed API's /health URL.", file=sys.stderr)
        return 2

    parsed = urlparse(base_url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        print("ERROR: DEPLOY_HEALTH_URL must be a valid http(s) URL.", file=sys.stderr)
        return 2

    try:
        response = requests.get(base_url, timeout=20)
        print("Health endpoint HTTP status:", response.status_code)
        response.raise_for_status()
        payload = response.json()
    except (requests.RequestException, json.JSONDecodeError) as exc:
        print(f"DEPLOYMENT VALIDATION FAILED: {type(exc).__name__}: {exc}", file=sys.stderr)
        return 1

    if not isinstance(payload, dict) or payload.get("status") != "ok":
        print("DEPLOYMENT VALIDATION FAILED: health JSON did not contain status='ok'.", file=sys.stderr)
        return 1

    print("DEPLOYMENT VALIDATION PASSED")
    print(json.dumps(payload, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
"""
(SCRIPTS_DIR / "validate_deployment.py").write_text(validation_script.strip() + "\n", encoding="utf-8")
print("Created:", SCRIPTS_DIR / "validate_deployment.py")

## 7. Create a manual GitHub Actions deployment-validation workflow

After you deploy the API using a hosting service of your choice, add a repository secret named `DEPLOY_HEALTH_URL` containing the full deployed health endpoint, for example `https://YOUR-HOST.example/health`. Do not put a real URL in source code if it is private.

This workflow validates an existing deployment. It does not provision hosting or upload the app, because that depends on the hosting provider you choose.

In [ ]:
validation_workflow = r"""
name: Validate Deployed RAG API

on:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  health-check:
    runs-on: ubuntu-latest
    timeout-minutes: 5
    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install validation dependency
        run: python -m pip install requests

      - name: Validate deployed health endpoint
        env:
          DEPLOY_HEALTH_URL: ${{ secrets.DEPLOY_HEALTH_URL }}
        run: python scripts/validate_deployment.py
"""
(WORKFLOW_DIR / "deployment-validation.yml").write_text(validation_workflow.strip() + "\n", encoding="utf-8")
print("Created:", WORKFLOW_DIR / "deployment-validation.yml")

## 8. Write the release and rollback plan

A rollback means returning to the last known-good version if a release fails validation. The exact commands to switch production versions depend on the hosting provider; use the platform's previous-release/rollback feature or redeploy the last known-good Git commit. Do not delete the last working release before the new one passes health checks.

In [ ]:
rollback_doc = r"""
# Release, Deployment Validation, and Rollback Plan

## Release checklist
1. Confirm the latest change is committed to Git.
2. Push the branch and open a pull request.
3. Wait for the `Customer Support RAG CI` workflow to pass.
4. Review the changed files and test results before merging.
5. Deploy through the chosen hosting provider's documented process.
6. Set the GitHub repository secret `DEPLOY_HEALTH_URL` to the deployed API's `/health` URL.
7. In GitHub, open **Actions → Validate Deployed RAG API → Run workflow**.
8. Confirm the job reports `DEPLOYMENT VALIDATION PASSED`.
9. Manually test `/docs`, `/query`, `/feedback`, and the Streamlit UI.

## Rollback triggers
- The health check fails or returns a non-200 response.
- `/query` repeatedly returns server errors.
- The UI cannot reach the API.
- A critical regression is observed after deployment.

## Rollback procedure
1. Stop promoting the current release; record the failed commit and error.
2. Use the hosting provider's **rollback/previous release** feature if available.
3. Otherwise redeploy the last known-good Git commit/tag using that provider's documented deployment process.
4. Re-run the `Validate Deployed RAG API` workflow.
5. Manually verify `/health`, `/docs`, one safe test query, feedback submission, and the Streamlit UI.
6. Keep logs and screenshots for the incident; fix the issue on a branch and let CI pass before trying again.

## Git-only recovery (for local source, not a production deploy)
- Inspect changes: `git status` and `git log --oneline -5`.
- To undo an unpushed local commit: `git reset --hard <last-known-good-commit>` (destructive; only if you understand that uncommitted changes will be lost).
- For a commit already shared with others, prefer `git revert <bad-commit>` and push the new revert commit. This preserves history.

**Important:** Git commands alone do not roll back a running remote service. The last step must use the hosting provider's deployment/release mechanism.

## Security reminders
- Never store credentials in source files or commit `.env`.
- Use repository/environment secrets for deployment settings.
- Restrict who can deploy and roll back production.
- Do not send customer personal information to logs or test fixtures.
- The demo API should be protected with authentication, rate limiting, and safe logging before public production use.
"""
(REPORT_DIR / "phase6_rollback_and_release_plan.md").write_text(rollback_doc.strip() + "\n", encoding="utf-8")
print("Created:", REPORT_DIR / "phase6_rollback_and_release_plan.md")

## 9. Create the final capstone report template

Use this report to explain what you actually implemented. Fill in the evaluation values from your Phase 3/4 CSV files; do not invent performance numbers. Include limitations such as the small synthetic knowledge base and any local-only parts.

In [ ]:
report_template = r"""
# Capstone Project 2: Agentic RAG for Enterprise Customer Support

**Author:** [Your name]  
**Date:** [Submission date]

## 1. Problem statement and use case
Describe the customer-support problem, expected users, and how grounded answers reduce repetitive support work.

## 2. Dataset and knowledge base
- Source: synthetic customer-support FAQ/policy knowledge base created for this capstone.
- Number of chunks: [fill in verified count]
- Chunking strategy and metadata: [describe from Phase 1]
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector store: FAISS

## 3. Retrieval design
Explain semantic retrieval, BM25 keyword retrieval, and hybrid rank fusion. Add example queries and retrieved source IDs. Mention whether a reranker was implemented or left as future work.

## 4. Baseline RAG and evaluation
Summarize Phase 3 results from `phase3_baseline_evaluation.csv` and `phase3_rouge_scores.csv`. Include ROUGE/BLEU/relevance only if actually calculated. Discuss manual relevance review and limitations of a five-question evaluation set.

## 5. Agentic workflow
Describe LangGraph nodes, planner, retrieval tool, answer generation, validation, retry/query rewrite, state/memory, and workflow traces. Add the LangGraph diagram and Phase 4 evaluation results.

## 6. Feedback loop
Explain thumbs-up/thumbs-down feedback, saved feedback fields, validation failures, retry limit, and how feedback could inform future tuning. State clearly what is implemented versus proposed.

## 7. API and user interface
Describe `GET /health`, `POST /query`, and `POST /feedback`. Include one real request/response example from `/docs` and screenshots of Streamlit.

## 8. CI/CD and deployment validation
Describe GitHub Actions triggers, Python version, Ruff checks, pytest, manual deployed-health validation, secret configuration, and rollback procedure. Include a passing Actions screenshot. State which hosting provider was used, if any; do not claim a public deployment if only tested locally.

## 9. Results and limitations
Compare baseline and agentic outputs using actual metrics and manual review. Discuss local model latency, synthetic/small corpus, source-grounding limits, and deployment/security limitations.

## 10. Future work
Potential work: larger verified corpus, reranking, richer relevance/faithfulness metrics, authentication, monitoring, human escalation, and provider-specific automated deployment.

## 11. Conclusion
Summarize what works, how to reproduce it, and what remains.

## 12. References
List model cards, library documentation, and any external dataset/document sources actually used.
"""
(REPORT_DIR / "phase6_project_report_template.md").write_text(report_template.strip() + "\n", encoding="utf-8")
print("Created:", REPORT_DIR / "phase6_project_report_template.md")

## 10. Create the screenshot/evidence checklist

In [ ]:
evidence = r"""
# Capstone Evidence Checklist

Capture genuine screenshots from your own project and GitHub repository. Mark each item only after it exists.

- [ ] Phase 2: FAISS/vector-store artifact creation and reload output.
- [ ] Phase 3: baseline evaluation table and ROUGE output.
- [ ] Phase 4: LangGraph workflow diagram.
- [ ] Phase 4: agent trace output and agentic evaluation table.
- [ ] Phase 5: FastAPI `/docs` showing `/health`, `/query`, and `/feedback`.
- [ ] Phase 5: successful `/health` request.
- [ ] Phase 5: `/query` request showing answer and source references.
- [ ] Phase 5: Streamlit customer-support chat UI with sources expanded.
- [ ] Phase 5: feedback submission saved successfully.
- [ ] Phase 6: GitHub Actions CI run showing Ruff and pytest passed.
- [ ] Phase 6: deployed endpoint health-check run, if an external deployment was completed.
- [ ] Phase 6: report section describing rollback and the actual hosting method.

Store screenshots in `reports/screenshots/`. Do not use fabricated or edited screenshots as proof of a passing workflow.
"""
(REPORT_DIR / "phase6_final_evidence_checklist.md").write_text(evidence.strip() + "\n", encoding="utf-8")
(PROJECT_ROOT / "reports" / "screenshots").mkdir(parents=True, exist_ok=True)
print("Created evidence checklist and reports/screenshots folder.")

## 11. Check the generated Python files for syntax errors

In [ ]:
import ast

files_to_check = [
    PROJECT_ROOT / "src" / "api.py",
    PROJECT_ROOT / "src" / "rag_service.py",
    PROJECT_ROOT / "src" / "streamlit_app.py",
    TEST_DIR / "test_api.py",
    SCRIPTS_DIR / "validate_deployment.py",
]
for path in files_to_check:
    if path.exists():
        ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
        print("Syntax OK:", path.relative_to(PROJECT_ROOT))
    else:
        print("WARNING - file not found:", path.relative_to(PROJECT_ROOT))

print("\nGenerated workflow files:")
for path in sorted(WORKFLOW_DIR.glob("*.yml")):
    print("-", path.relative_to(PROJECT_ROOT))

## 12. Run tests locally

Run this from the project root in the VS Code terminal with `.venv` activated. The first command installs Ruff if needed. These tests should not download the ML models.

In [ ]:
print("From the VS Code terminal, run these commands:")
print("  python -m pip install ruff pytest httpx")
print("  python -m ruff check src tests")
print("  python -m pytest -q")

## 13. GitHub setup — beginner steps

1. Create a repository on GitHub (or use your existing one).
2. In the VS Code terminal, run `git status` and make sure `.venv` and secrets are not tracked.
3. Add a `.gitignore` if needed. At minimum ignore `.venv/`, `__pycache__/`, `.env`, `.pytest_cache/`, `.ruff_cache/`, and generated logs containing user data.
4. Commit and push the project files, including `.github/workflows/ci.yml` and tests.
5. Open GitHub → **Actions**. The CI workflow should run on push/PR.
6. Open the latest `Customer Support RAG CI` run and inspect the Ruff and pytest steps.
7. Fix failures locally, commit, and push again. Do not mark the workflow as passing until GitHub shows a green run.
8. If you later deploy the API publicly, add `DEPLOY_HEALTH_URL` in **Settings → Secrets and variables → Actions → New repository secret**. Then manually run **Validate Deployed RAG API**.

### Git commands (run in the project root)
```bash
git status
git add .github/workflows tests scripts reports pyproject.toml
git commit -m "Add CI, deployment validation, and rollback plan"
git push
```

If Git is not initialized yet, run `git init -b main` first, then add the GitHub remote URL supplied by GitHub. Never paste credentials into the notebook.

## 14. Final validation

This confirms the notebook created the Phase 6 files locally. It does not prove GitHub Actions passed or that an external deployment exists; those must be verified in GitHub and on your hosting provider.

In [ ]:
required_files = [
    WORKFLOW_DIR / "ci.yml",
    WORKFLOW_DIR / "deployment-validation.yml",
    TEST_DIR / "test_api.py",
    SCRIPTS_DIR / "validate_deployment.py",
    REPORT_DIR / "phase6_rollback_and_release_plan.md",
    REPORT_DIR / "phase6_project_report_template.md",
    REPORT_DIR / "phase6_final_evidence_checklist.md",
    PROJECT_ROOT / "pyproject.toml",
]
missing = [str(p.relative_to(PROJECT_ROOT)) for p in required_files if not p.exists()]
for path in required_files:
    print(("OK: " if path.exists() else "MISSING: ") + str(path.relative_to(PROJECT_ROOT)))
assert not missing, f"Missing Phase 6 files: {missing}"
print("\nPHASE 6 CI/CD AND DEPLOYMENT-VALIDATION FILES CREATED SUCCESSFULLY.")
print("Next: run pytest locally, push to GitHub, and inspect the Actions workflow result.")

## What is complete and what is still required

**Created locally by this notebook:** CI workflow, model-free API tests, lint config, deployed health-check script/workflow, report template, evidence checklist, and rollback plan.

**You must still do manually:** run tests locally, push to GitHub, verify the workflow actually passes, choose a hosting provider if public deployment is required, configure the health URL secret, run deployment validation, collect real screenshots, and fill the report with actual evaluation values.

**No containerization files or container build/publish steps are included.**